# Preprocessing Data Galaxy Form Burst

Notebook ini melakukan preprocessing lengkap pada dataset `sdss_100k_galaxy_form_burst.csv`, menghasilkan dua dataset terpisah:
- **`dataset_klasifikasi.csv`** — Dataset seimbang 10.000 baris (5.000 STARFORMING + 5.000 STARBURST) dengan 10 fitur terbaik untuk klasifikasi.
- **`dataset_regresi.csv`** — Dataset 10.000 baris acak dengan 10 fitur terbaik untuk regresi redshift.

Pipeline seleksi fitur menggunakan **Gini Importance** (klasifikasi) dan **SSR Reduction** (regresi), konsisten dengan implementasi CART di proyek ini.

---
## 1. Import Library
Mengimpor semua library yang dibutuhkan.

In [1]:
import math
import random
import pandas as pd
from collections import Counter

---
## 2. Load Dataset
Membaca dataset mentah dari file CSV ke dalam DataFrame pandas.

In [2]:
df_raw = pd.read_csv("sdss_100k_galaxy_form_burst.csv")
print(f"Shape dataset mentah: {df_raw.shape}")
print(f"Kolom tersedia: {list(df_raw.columns)}")
df_raw.head()

Shape dataset mentah: (100000, 43)
Kolom tersedia: ['objid', 'specobjid', 'ra', 'dec', 'u', 'g', 'r', 'i', 'z', 'modelFlux_u', 'modelFlux_g', 'modelFlux_r', 'modelFlux_i', 'modelFlux_z', 'petroRad_u', 'petroRad_g', 'petroRad_i', 'petroRad_r', 'petroRad_z', 'petroFlux_u', 'petroFlux_g', 'petroFlux_i', 'petroFlux_r', 'petroFlux_z', 'petroR50_u', 'petroR50_g', 'petroR50_i', 'petroR50_r', 'petroR50_z', 'psfMag_u', 'psfMag_r', 'psfMag_g', 'psfMag_i', 'psfMag_z', 'expAB_u', 'expAB_g', 'expAB_r', 'expAB_i', 'expAB_z', 'class', 'subclass', 'redshift', 'redshift_err']


,objid,specobjid,ra,dec,u,g,r,i,z,modelFlux_u,...,psfMag_z,expAB_u,expAB_g,expAB_r,expAB_i,expAB_z,class,subclass,redshift,redshift_err
0,1.240000e+18,8.180000e+18,82.038679,0.847177,21.73818,20.26633,19.32409,18.64037,18.23833,2.007378,...,19.43575,0.099951,0.311864,0.289370,0.270588,0.187182,GALAXY,STARFORMING,0.067749,0.000015
1,1.240000e+18,8.180000e+18,82.138894,1.063072,20.66761,19.32016,18.67888,18.24693,18.04122,5.403369,...,18.85012,0.366549,0.516876,0.517447,0.552297,0.636966,GALAXY,STARFORMING,0.105118,0.000010
2,1.240000e+18,8.180000e+18,82.028510,1.104003,23.63531,21.19671,19.92297,19.31443,18.68396,0.295693,...,19.42235,0.050000,0.417137,0.506950,0.549881,0.370166,GALAXY,STARFORMING,0.234089,0.000030
3,1.240000e+18,3.320000e+17,198.544469,-1.097059,20.12374,18.41520,17.47202,17.05297,16.72423,8.920645,...,18.03204,0.310763,0.356827,0.389345,0.388160,0.416660,GALAXY,STARFORMING,0.110825,0.000030
4,1.240000e+18,3.320000e+17,198.706863,-1.046217,-9999.00000,-9999.00000,18.37762,18.13383,17.78497,0.000000,...,19.02880,-9999.000000,-9999.000000,0.050000,0.050000,0.149973,GALAXY,STARFORMING,0.136658,0.000021


---
## 3. Pembersihan Data Global

### 3a. Drop Kolom Identitas & Koordinat
Menghapus kolom yang tidak relevan sebagai fitur:
- `objid`, `specobjid` — ID unik objek
- `ra`, `dec` — Koordinat langit
- `class` — Label kelas tingkat atas (semua isinya `GALAXY`, tidak informatif)

Kolom **`subclass`** (STARFORMING/STARBURST) dan **`redshift`** **dipertahankan** karena masing-masing merupakan target untuk dataset klasifikasi dan regresi.

In [3]:
# Kolom yang dihapus: ID, koordinat, dan class tingkat atas
columns_to_drop = ["objid", "specobjid", "ra", "dec", "class"]

df_clean = df_raw.drop(columns=columns_to_drop)
print(f"Setelah drop kolom identitas: {df_clean.shape}")

Setelah drop kolom identitas: (100000, 38)


### 3b. Hapus Baris dengan Placeholder Null (-9999.00)
Dataset menggunakan nilai `-9999.00` sebagai penanda missing value. Seluruh baris yang mengandung nilai ini dihapus.

In [4]:
# Hapus semua baris yang memiliki nilai placeholder -9999.00 pada kolom manapun
df_clean = df_clean[df_clean.ne(-9999.00).all(axis=1)]
df_clean = df_clean.reset_index(drop=True)

print(f"Setelah hapus -9999 placeholder: {df_clean.shape}")
print(f"Distribusi subclass:\n{df_clean['subclass'].value_counts()}")
df_clean.head()

Setelah hapus -9999 placeholder: (97478, 38)
Distribusi subclass:
subclass
STARFORMING    73518
STARBURST      23960
Name: count, dtype: int64


,u,g,r,i,z,modelFlux_u,modelFlux_g,modelFlux_r,modelFlux_i,modelFlux_z,...,psfMag_i,psfMag_z,expAB_u,expAB_g,expAB_r,expAB_i,expAB_z,subclass,redshift,redshift_err
0,21.73818,20.26633,19.32409,18.64037,18.23833,2.007378,7.823640,18.63581,34.98175,50.64961,...,20.07646,19.43575,0.099951,0.311864,0.289370,0.270588,0.187182,STARFORMING,0.067749,0.000015
1,20.66761,19.32016,18.67888,18.24693,18.04122,5.403369,18.703640,33.76298,50.25997,60.73625,...,19.19277,18.85012,0.366549,0.516876,0.517447,0.552297,0.636966,STARFORMING,0.105118,0.000010
2,23.63531,21.19671,19.92297,19.31443,18.68396,0.295693,3.318924,10.73388,18.80136,33.58972,...,20.00731,19.42235,0.050000,0.417137,0.506950,0.549881,0.370166,STARFORMING,0.234089,0.000030
3,20.12374,18.41520,17.47202,17.05297,16.72423,8.920645,43.044740,102.61010,150.94260,204.31610,...,18.38868,18.03204,0.310763,0.356827,0.389345,0.388160,0.416660,STARFORMING,0.110825,0.000030
4,19.47473,18.18575,17.52763,17.14837,16.89580,16.220930,53.173800,97.48736,138.24510,174.45070,...,18.44931,18.23220,0.754158,0.767767,0.759105,0.742471,0.721491,STARFORMING,0.111458,0.000011


---
## 4. Definisi Daftar Fitur

Mendefinisikan semua kolom fitur numerik yang akan digunakan sebagai kandidat fitur input untuk kedua model. Kolom `subclass` dan `redshift` adalah target, bukan fitur.

In [5]:
# Semua kolom fitur numerik (kandidat input)
FEATURE_COLS = [
    'u', 'g', 'r', 'i', 'z',
    'modelFlux_u', 'modelFlux_g', 'modelFlux_r', 'modelFlux_i', 'modelFlux_z',
    'petroRad_u',  'petroRad_g',  'petroRad_r',  'petroRad_i',  'petroRad_z',
    'petroFlux_u', 'petroFlux_g', 'petroFlux_r', 'petroFlux_i', 'petroFlux_z',
    'petroR50_u',  'petroR50_g',  'petroR50_r',  'petroR50_i',  'petroR50_z',
    'psfMag_u',    'psfMag_g',    'psfMag_r',    'psfMag_i',    'psfMag_z',
    'expAB_u',     'expAB_g',     'expAB_r',     'expAB_i',     'expAB_z',
]

print(f"Jumlah kandidat fitur: {len(FEATURE_COLS)}")

Jumlah kandidat fitur: 35


---
## 5. Pemisahan Dataset untuk Klasifikasi

### 5a. Pemisahan Kelas (Class Separation)
Memisahkan dataset bersih menjadi dua kelompok berdasarkan kolom `subclass`:
- `df_starforming` — Hanya baris dengan label `STARFORMING`
- `df_starburst` — Hanya baris dengan label `STARBURST`

In [6]:
# Pisahkan berdasarkan label subclass
df_starforming = df_clean[df_clean["subclass"] == "STARFORMING"].reset_index(drop=True)
df_starburst   = df_clean[df_clean["subclass"] == "STARBURST"].reset_index(drop=True)

print(f"Jumlah STARFORMING: {len(df_starforming):,} baris")
print(f"Jumlah STARBURST  : {len(df_starburst):,} baris")

Jumlah STARFORMING: 73,518 baris
Jumlah STARBURST  : 23,960 baris


### 5b. Balanced Sampling 5.000 + 5.000
Mengambil **5.000 baris secara acak** dari masing-masing kelas, lalu menggabungkannya menjadi satu dataset seimbang dengan **tepat 10.000 baris**.

- `random_state=42` digunakan agar hasil dapat direproduksi.

In [7]:
SAMPLE_SIZE    = 5000
RANDOM_STATE   = 42

# Ambil 5.000 baris acak dari tiap kelas
df_sf_sample = df_starforming.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)
df_sb_sample = df_starburst.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

# Gabungkan dan acak urutan baris
df_klasifikasi = pd.concat([df_sf_sample, df_sb_sample], ignore_index=True)
df_klasifikasi = df_klasifikasi.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Shape dataset klasifikasi: {df_klasifikasi.shape}")
print(f"Distribusi kelas (harus seimbang):\n{df_klasifikasi['subclass'].value_counts()}")
df_klasifikasi.head()

Shape dataset klasifikasi: (10000, 38)
Distribusi kelas (harus seimbang):
subclass
STARBURST      5000
STARFORMING    5000
Name: count, dtype: int64


,u,g,r,i,z,modelFlux_u,modelFlux_g,modelFlux_r,modelFlux_i,modelFlux_z,...,psfMag_i,psfMag_z,expAB_u,expAB_g,expAB_r,expAB_i,expAB_z,subclass,redshift,redshift_err
0,20.84262,20.10340,19.58965,19.35887,19.04977,4.597819,9.090663,14.59189,18.04716,23.97061,...,19.49746,19.16715,0.050000,0.999924,0.999898,0.999742,0.999803,STARBURST,0.249136,0.000009
1,19.32499,17.98579,17.32878,16.97005,16.72728,18.620000,63.926560,117.08080,162.92240,203.74330,...,18.62543,18.20098,0.816894,0.808926,0.829465,0.834572,0.805672,STARFORMING,0.046189,0.000016
2,20.14565,18.47789,17.57100,17.07108,16.81810,8.742394,40.629470,93.66942,148.44630,187.39240,...,19.17272,18.82300,0.102062,0.365911,0.373200,0.405507,0.426919,STARFORMING,0.185548,0.000019
3,19.43511,18.21046,17.71781,17.46296,17.28559,16.823880,51.977550,81.82313,103.46990,121.82820,...,19.59073,19.33015,0.731971,0.889433,0.880142,0.904787,0.890327,STARFORMING,0.123105,0.000013
4,20.41864,18.67497,17.67722,17.16472,16.78192,6.797657,33.884990,84.94001,136.17890,193.74230,...,18.14523,17.94823,0.330693,0.509506,0.494502,0.505395,0.507876,STARFORMING,0.128123,0.000024


---
## 6. Pembuatan Dataset untuk Regresi

### Downsampling Representatif 10.000 Baris
Mengambil **10.000 baris secara acak** langsung dari dataset bersih global tanpa memandang subkelas. Dataset ini digunakan untuk memprediksi nilai kontinu `redshift`.

Kolom `subclass` tidak diperlukan dalam dataset regresi.

In [8]:
REGRESSION_SAMPLE_SIZE = 10000

# Ambil 10.000 baris acak dari dataset bersih (tanpa memandang subkelas)
df_regresi = df_clean.sample(n=REGRESSION_SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

# Hapus kolom subclass karena tidak relevan untuk regresi
df_regresi = df_regresi.drop(columns=["subclass"])

print(f"Shape dataset regresi: {df_regresi.shape}")
print(f"Statistik target redshift:\n{df_regresi['redshift'].describe()}")
df_regresi.head()

Shape dataset regresi: (10000, 37)
Statistik target redshift:
count    10000.000000
mean         0.112615
std          0.093877
min         -0.000619
25%          0.055765
50%          0.084804
75%          0.132422
max          0.561523
Name: redshift, dtype: float64


,u,g,r,i,z,modelFlux_u,modelFlux_g,modelFlux_r,modelFlux_i,modelFlux_z,...,psfMag_g,psfMag_i,psfMag_z,expAB_u,expAB_g,expAB_r,expAB_i,expAB_z,redshift,redshift_err
0,19.53920,18.49463,17.80479,17.44611,17.24578,15.285630,40.00774,75.52381,105.08880,126.37800,...,20.24937,19.14744,18.82794,0.762037,0.911514,0.952104,0.968609,0.860584,0.146655,0.000010
1,20.04008,18.63083,17.94050,17.52629,17.24970,9.635576,35.29115,66.64987,97.60743,125.92250,...,20.02224,18.88761,18.63392,0.277009,0.342071,0.355511,0.333149,0.373133,0.063396,0.000013
2,18.88969,17.45173,16.67817,16.19766,15.88523,27.804300,104.54590,213.17330,331.84490,442.49170,...,19.13179,17.74829,17.39477,0.281752,0.340521,0.362044,0.340654,0.345006,0.076058,0.000006
3,20.05519,19.39178,18.70634,18.31313,18.11321,9.502275,17.50965,32.91989,47.28717,56.83823,...,19.92417,19.05531,18.77191,0.050000,0.587002,0.621163,0.578835,0.685937,0.231372,0.000013
4,18.13070,16.77617,16.11394,15.69656,15.39237,55.939300,194.77380,358.44510,526.47330,696.70850,...,19.12746,18.02605,17.68450,0.385111,0.345675,0.351770,0.371869,0.378457,0.071396,0.000007


---
## 7. Seleksi Fitur Terbaik (Best Feature Selection)

Mengimplementasikan dua metode seleksi fitur yang konsisten dengan logika CART di proyek ini:

| Dataset | Metode | Ukuran Kemurnian |
|---------|--------|------------------|
| Klasifikasi | **Gini Importance** | Penurunan Gini Impurity |
| Regresi | **SSR Reduction** | Penurunan Sum of Squared Residuals |

Setiap fitur diuji secara individual menggunakan **Decision Stump** (pohon 1 level), kemudian diranking berdasarkan gain terbesar. **Top 10 fitur** dipilih untuk masing-masing dataset.

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Fungsi Bantuan Umum
# ─────────────────────────────────────────────────────────────────────────────

def find_best_threshold(values_col, labels):
    """
    Mencari threshold optimal dari satu fitur menggunakan midpoint dari
    semua nilai unik yang terurut.
    Mengembalikan (best_threshold, best_gain, gain_fn) — gain_fn dipilih
    berdasarkan tipe label (string → Gini, numerik → SSR).
    """
    values = sorted(set(values_col))
    if len(values) <= 1:
        return None, 0.0
    thresholds = [(values[i] + values[i+1]) / 2 for i in range(len(values)-1)]
    return thresholds

# ─────────────────────────────────────────────────────────────────────────────
# Fungsi Gini Impurity (untuk Klasifikasi)
# ─────────────────────────────────────────────────────────────────────────────

def gini(labels):
    """Gini Impurity = 1 - Σ(p_i²). Nilai 0 berarti murni."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = Counter(labels)
    return 1.0 - sum((c / n) ** 2 for c in counts.values())

def gini_gain(parent_labels, left_labels, right_labels):
    """Weighted Gini Gain = Gini(parent) - weighted Gini(children)."""
    n = len(left_labels) + len(right_labels)
    child_gini = (len(left_labels) / n) * gini(left_labels) + \
                 (len(right_labels) / n) * gini(right_labels)
    return gini(parent_labels) - child_gini

def best_feature_gini(feature_values, labels, n_sample=500):
    """
    Menghitung Gini Gain terbaik untuk satu fitur menggunakan Decision Stump.
    Menggunakan sampling threshold untuk efisiensi komputasi.
    """
    thresholds = find_best_threshold(feature_values, labels)
    if thresholds is None:
        return 0.0
    # Sub-sample threshold jika terlalu banyak
    if len(thresholds) > n_sample:
        thresholds = random.sample(thresholds, n_sample)
    
    best_gain = 0.0
    parent_labels = labels
    data_pairs = list(zip(feature_values, labels))
    
    for thresh in thresholds:
        left_labels  = [y for x, y in data_pairs if x <= thresh]
        right_labels = [y for x, y in data_pairs if x > thresh]
        if not left_labels or not right_labels:
            continue
        gain = gini_gain(parent_labels, left_labels, right_labels)
        if gain > best_gain:
            best_gain = gain
    return best_gain


# ─────────────────────────────────────────────────────────────────────────────
# Fungsi SSR Reduction (untuk Regresi)
# ─────────────────────────────────────────────────────────────────────────────

def ssr(labels):
    """Sum of Squared Residuals = Σ(y_i - mean(y))²."""
    if len(labels) == 0:
        return 0.0
    m = sum(labels) / len(labels)
    return sum((v - m) ** 2 for v in labels)

def ssr_gain(parent_labels, left_labels, right_labels):
    """SSR Gain = SSR(parent) - [SSR(left) + SSR(right)]."""
    return ssr(parent_labels) - (ssr(left_labels) + ssr(right_labels))

def best_feature_ssr(feature_values, labels, n_sample=500):
    """
    Menghitung SSR Gain terbaik untuk satu fitur menggunakan Decision Stump.
    Menggunakan sampling threshold untuk efisiensi komputasi.
    """
    thresholds = find_best_threshold(feature_values, labels)
    if thresholds is None:
        return 0.0
    if len(thresholds) > n_sample:
        thresholds = random.sample(thresholds, n_sample)
    
    best_gain = 0.0
    parent_labels = labels
    data_pairs = list(zip(feature_values, labels))
    
    for thresh in thresholds:
        left_labels  = [y for x, y in data_pairs if x <= thresh]
        right_labels = [y for x, y in data_pairs if x > thresh]
        if not left_labels or not right_labels:
            continue
        gain = ssr_gain(parent_labels, left_labels, right_labels)
        if gain > best_gain:
            best_gain = gain
    return best_gain


print("Fungsi seleksi fitur berhasil didefinisikan.")

Fungsi seleksi fitur berhasil didefinisikan.


---
## 8. Seleksi Top 10 Fitur — Dataset Klasifikasi

Menggunakan **Gini Gain** sebagai metrik. Setiap fitur dari `FEATURE_COLS` diuji secara individual menggunakan Decision Stump, lalu diurutkan dari gain tertinggi ke terendah. **10 fitur teratas** dipilih.

In [11]:
random.seed(RANDOM_STATE)

print("Menghitung Gini Gain untuk setiap fitur (dataset klasifikasi)...")
print("-" * 50)

# Label target klasifikasi
y_klasifikasi = df_klasifikasi["subclass"].tolist()

gini_scores = {}
for feat in FEATURE_COLS:
    feat_values = df_klasifikasi[feat].tolist()
    score = best_feature_gini(feat_values, y_klasifikasi)
    gini_scores[feat] = score

# Urutkan dari gain tertinggi
gini_scores_sorted = sorted(gini_scores.items(), key=lambda x: x[1], reverse=True)

print(f"{'Rank':<5} {'Fitur':<20} {'Gini Gain'}")
print("-" * 40)
for rank, (feat, score) in enumerate(gini_scores_sorted, 1):
    marker = " ← TOP 10" if rank <= 10 else ""
    print(f"{rank:<5} {feat:<20} {score:.6f}{marker}")

Menghitung Gini Gain untuk setiap fitur (dataset klasifikasi)...
--------------------------------------------------
Rank  Fitur                Gini Gain
----------------------------------------
1     petroR50_u           0.106951 ← TOP 10
2     petroRad_g           0.105745 ← TOP 10
3     petroR50_g           0.104113 ← TOP 10
4     petroRad_r           0.098980 ← TOP 10
5     petroR50_r           0.096459 ← TOP 10
6     modelFlux_z          0.094339 ← TOP 10
7     z                    0.094307 ← TOP 10
8     petroR50_i           0.094060 ← TOP 10
9     petroRad_i           0.092733 ← TOP 10
10    petroFlux_z          0.088287 ← TOP 10
11    petroR50_z           0.085278
12    petroRad_u           0.085229
13    i                    0.084995
14    modelFlux_i          0.084995
15    petroFlux_i          0.081212
16    r                    0.079587
17    modelFlux_r          0.079541
18    petroFlux_r          0.079034
19    petroRad_z           0.075028
20    petroFlux_g          0.060

In [12]:
# Ambil top 10 fitur klasifikasi
TOP_10_KLASIFIKASI = [feat for feat, _ in gini_scores_sorted[:10]]

print(f"Top 10 fitur untuk KLASIFIKASI:")
for i, f in enumerate(TOP_10_KLASIFIKASI, 1):
    print(f"  {i:2d}. {f}")

Top 10 fitur untuk KLASIFIKASI:
   1. petroR50_u
   2. petroRad_g
   3. petroR50_g
   4. petroRad_r
   5. petroR50_r
   6. modelFlux_z
   7. z
   8. petroR50_i
   9. petroRad_i
  10. petroFlux_z


---
## 9. Seleksi Top 10 Fitur — Dataset Regresi

Menggunakan **SSR Reduction** sebagai metrik. Setiap fitur diuji secara individual, diranking dari pengurangan SSR terbesar. **10 fitur teratas** dipilih.

In [13]:
random.seed(RANDOM_STATE)

print("Menghitung SSR Gain untuk setiap fitur (dataset regresi)...")
print("-" * 50)

# Label target regresi
y_regresi = df_regresi["redshift"].tolist()

ssr_scores = {}
for feat in FEATURE_COLS:
    feat_values = df_regresi[feat].tolist()
    score = best_feature_ssr(feat_values, y_regresi)
    ssr_scores[feat] = score

# Urutkan dari gain tertinggi
ssr_scores_sorted = sorted(ssr_scores.items(), key=lambda x: x[1], reverse=True)

print(f"{'Rank':<5} {'Fitur':<20} {'SSR Gain'}")
print("-" * 45)
for rank, (feat, score) in enumerate(ssr_scores_sorted, 1):
    marker = " ← TOP 10" if rank <= 10 else ""
    print(f"{rank:<5} {feat:<20} {score:.4f}{marker}")

Menghitung SSR Gain untuk setiap fitur (dataset regresi)...
--------------------------------------------------
Rank  Fitur                SSR Gain
---------------------------------------------
1     g                    34.9596 ← TOP 10
2     modelFlux_g          34.9545 ← TOP 10
3     petroFlux_g          34.8698 ← TOP 10
4     u                    34.0925 ← TOP 10
5     modelFlux_u          34.0346 ← TOP 10
6     r                    31.2187 ← TOP 10
7     modelFlux_r          31.2013 ← TOP 10
8     psfMag_g             30.9901 ← TOP 10
9     petroFlux_r          30.9252 ← TOP 10
10    petroFlux_u          30.8719 ← TOP 10
11    modelFlux_i          30.1873
12    petroFlux_i          30.1788
13    i                    30.1293
14    modelFlux_z          28.4566
15    z                    28.4103
16    petroFlux_z          27.9998
17    psfMag_u             23.0712
18    psfMag_r             21.2452
19    petroRad_g           19.7571
20    petroR50_r           18.9809
21    petroR50_i 

In [14]:
# Ambil top 10 fitur regresi
TOP_10_REGRESI = [feat for feat, _ in ssr_scores_sorted[:10]]

print(f"Top 10 fitur untuk REGRESI:")
for i, f in enumerate(TOP_10_REGRESI, 1):
    print(f"  {i:2d}. {f}")

Top 10 fitur untuk REGRESI:
   1. g
   2. modelFlux_g
   3. petroFlux_g
   4. u
   5. modelFlux_u
   6. r
   7. modelFlux_r
   8. psfMag_g
   9. petroFlux_r
  10. petroFlux_u


---
## 10. Validasi: Tree-Based Feature Importance (Sklearn)

Sebagai **metode pembanding**, digunakan `sklearn.tree` untuk menghitung **Feature Importance berbasis pohon keputusan** yang sudah ter-fit pada seluruh dataset.

| | Metode Manual (Section 8-9) | Metode Sklearn (Section ini) |
|---|---|---|
| **Cara kerja** | Decision Stump per-fitur, gain dihitung individual | Decision Tree penuh (max_depth=5) yang difit sekaligus |
| **Scope** | Univariat — satu fitur vs target | Multivariat — semua fitur dipertimbangkan bersama |
| **Output** | Gain absolut | Proporsi impurity reduction (sum = 1.0) |
| **Keunggulan** | Tanpa dependensi sklearn | Menangkap interaksi antar-fitur |

Hasil kedua metode dapat dibandingkan untuk melihat konsistensi ranking fitur.

In [17]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Load ulang dan bersihkan placeholder -9999 menggunakan np.nan agar dropna() bisa bekerja
df_sk = pd.read_csv("sdss_100k_galaxy_form_burst.csv")
df_sk = df_sk.replace(-9999.00, np.nan).dropna()

# Buang kolom non-fisis / ID — sama seperti pipeline manual di atas
drop_sk = ["objid", "specobjid", "ra", "dec", "class", "subclass", "redshift"]
if "redshift_err" in df_sk.columns:
    drop_sk.append("redshift_err")

X_raw = df_sk.drop(columns=drop_sk)

print(f"Dataset untuk sklearn: {df_sk.shape}")
print(f"Fitur kandidat       : {X_raw.shape[1]} kolom")

# ==========================================
# A. KLASIFIKASI — Top 10 Fitur untuk Subclass (STARFORMING / STARBURST)
# ==========================================
y_class = df_sk["subclass"]

model_class = DecisionTreeClassifier(max_depth=5, random_state=42)
model_class.fit(X_raw, y_class)

# Gini Importance = rata-rata penurunan impurity tertimbang di semua node
importances_class = pd.Series(model_class.feature_importances_, index=X_raw.columns)
top10_class_sk = importances_class.nlargest(10)

print("\n" + "=" * 50)
print("TOP 10 FITUR — KLASIFIKASI SUBCLASS (sklearn)")
print("=" * 50)
print(top10_class_sk.to_string())

# ==========================================
# B. REGRESI — Top 10 Fitur untuk Redshift
# ==========================================
y_reg = df_sk["redshift"]

model_reg = DecisionTreeRegressor(max_depth=5, random_state=42)
model_reg.fit(X_raw, y_reg)

# SSR-based Importance = rata-rata penurunan MSE tertimbang di semua node
importances_reg = pd.Series(model_reg.feature_importances_, index=X_raw.columns)
top10_reg_sk = importances_reg.nlargest(10)

print("\n" + "=" * 50)
print("TOP 10 FITUR — REGRESI REDSHIFT (sklearn)")
print("=" * 50)
print(top10_reg_sk.to_string())

# ==========================================
# Perbandingan ranking: Manual vs Sklearn
# ==========================================
print("\n" + "=" * 58)
print("PERBANDINGAN RANKING FITUR — KLASIFIKASI")
print("=" * 58)
print(f"{'Rank':<5} {'Manual (Gini Stump)':<25} {'Sklearn (Tree Importance)':<25}")
print("-" * 58)
top10_sk_class_list = list(top10_class_sk.index)
for i in range(10):
    manual = TOP_10_KLASIFIKASI[i] if i < len(TOP_10_KLASIFIKASI) else "-"
    sk     = top10_sk_class_list[i] if i < len(top10_sk_class_list) else "-"
    match  = " " if manual == sk else " "
    print(f"{i+1:<5} {manual:<25} {sk:<25} {match}")

print("\n" + "=" * 58)
print("PERBANDINGAN RANKING FITUR — REGRESI")
print("=" * 58)
print(f"{'Rank':<5} {'Manual (SSR Stump)':<25} {'Sklearn (Tree Importance)':<25}")
print("-" * 58)
top10_sk_reg_list = list(top10_reg_sk.index)
for i in range(10):
    manual = TOP_10_REGRESI[i] if i < len(TOP_10_REGRESI) else "-"
    sk     = top10_sk_reg_list[i] if i < len(top10_sk_reg_list) else "-"
    match  = " " if manual == sk else " "
    print(f"{i+1:<5} {manual:<25} {sk:<25} {match}")


Dataset untuk sklearn: (97478, 43)
Fitur kandidat       : 35 kolom

TOP 10 FITUR — KLASIFIKASI SUBCLASS (sklearn)
petroRad_g     0.454474
psfMag_u       0.253331
z              0.153919
psfMag_z       0.042901
psfMag_r       0.042471
petroR50_i     0.023030
modelFlux_z    0.022308
petroFlux_u    0.002027
expAB_z        0.002006
expAB_i        0.001510

TOP 10 FITUR — REGRESI REDSHIFT (sklearn)
modelFlux_g    0.641610
psfMag_z       0.098874
psfMag_g       0.087590
petroFlux_g    0.082101
modelFlux_u    0.022771
g              0.013035
psfMag_r       0.012287
expAB_i        0.009437
expAB_r        0.006775
petroRad_u     0.005804

PERBANDINGAN RANKING FITUR — KLASIFIKASI
Rank  Manual (Gini Stump)       Sklearn (Tree Importance)
----------------------------------------------------------
1     petroR50_u                petroRad_g                 
2     petroRad_g                psfMag_u                   
3     petroR50_g                z                          
4     petroRad_r        

---
## 11. Finalisasi dan Simpan Dataset

Memilih hanya kolom top 10 fitur + kolom target, kemudian menyimpan ke CSV terpisah.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset Klasifikasi: top 10 fitur + subclass
# ─────────────────────────────────────────────────────────────────────────────
df_klasifikasi_final = df_klasifikasi[TOP_10_KLASIFIKASI + ["subclass"]]

print("=" * 55)
print("DATASET KLASIFIKASI")
print("=" * 55)
print(f"Shape : {df_klasifikasi_final.shape}")
print(f"Kolom : {list(df_klasifikasi_final.columns)}")
print(f"Distribusi subclass:\n{df_klasifikasi_final['subclass'].value_counts()}")
df_klasifikasi_final.head()

ValueError: Lengths of operands do not match: 1 != 10

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset Regresi: top 10 fitur + redshift
# ─────────────────────────────────────────────────────────────────────────────
df_regresi_final = df_regresi[top10_class_sk + ["redshift"]]

print("=" * 55)
print("DATASET REGRESI")
print("=" * 55)
print(f"Shape : {df_regresi_final.shape}")
print(f"Kolom : {list(df_regresi_final.columns)}")
print(f"Statistik redshift:\n{df_regresi_final['redshift'].describe()}")
df_regresi_final.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Simpan kedua dataset ke CSV
# ─────────────────────────────────────────────────────────────────────────────
df_klasifikasi_final.to_csv("dataset_klasifikasi.csv", index=False)
df_regresi_final.to_csv("dataset_regresi.csv", index=False)

print("✓ dataset_klasifikasi.csv berhasil disimpan.")
print(f"  → Shape: {df_klasifikasi_final.shape} | Fitur: {TOP_10_KLASIFIKASI}")
print()
print("✓ dataset_regresi.csv berhasil disimpan.")
print(f"  → Shape: {df_regresi_final.shape} | Fitur: {TOP_10_REGRESI}")

In [19]:
df_klasifikasi.to_csv("DataClass.csv", index=False)
df_regresi.to_csv("DataReg.csv", index=False)